# Representation Engineering

Extract reading vectors via PCA of contrastive pair activations,
apply control vector interventions, and score concept presence
across layers. Based on Zou et al. 2023 (RepE).

This notebook covers the `irtk.representation_engineering` module.

## Why This Matters

Representation engineering (RepE) extracts 'reading vectors' that detect specific concepts in activations, and 'control vectors' that steer model behavior when added to the residual stream. This bridges interpretability and control — understanding what the model represents enables targeted interventions.

**Key references:**
- [Zou et al. (2023) "Representation Engineering"](https://arxiv.org/abs/2310.01405) — Reading and controlling model behavior via representations
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import representation_engineering as repe

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Extract Reading Vectors

Find the direction in activation space that separates positive from
negative examples of a concept.

In [ ]:
# Positive: sentences about France/Paris
pos_prompts = [
    model.to_tokens("The Eiffel Tower is located in Paris, France"),
    model.to_tokens("France is a country in Europe with"),
    model.to_tokens("The capital of France is Paris which"),
]
# Negative: sentences about other places
neg_prompts = [
    model.to_tokens("The Statue of Liberty is in New York"),
    model.to_tokens("Japan is a country in Asia with"),
    model.to_tokens("The capital of Germany is Berlin which"),
]

result = repe.extract_reading_vectors(
    model, pos_prompts, neg_prompts, "blocks.8.hook_resid_post"
)

print(f"Separation score: {result['separation_score']:.4f}")
print(f"Explained variance: {result['explained_variance'][0]:.4f}")
print(f"Reading vector norm: {np.linalg.norm(result['reading_vectors'][0]):.4f}")

## 2. Reading Vector Scan

Where in the network is the concept best represented?

In [ ]:
scan = repe.reading_vector_scan(model, pos_prompts, neg_prompts)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.abs(scan['separation_scores']), 'bo-')
ax.axvline(scan['best_layer'], color='red', alpha=0.3, label=f'Best layer: {scan["best_layer"]}')
ax.set_xlabel('Layer')
ax.set_ylabel('|Separation Score|')
ax.set_title('Concept Encoding Across Layers')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Representation Scoring

Project activations onto the reading vector for per-token concept scores.

In [ ]:
test_tokens = model.to_tokens("The Eiffel Tower is a famous landmark in Paris")
best_hook = f"blocks.{scan['best_layer']}.hook_resid_post"
rv = scan['reading_vectors'][scan['best_layer']]

scores = repe.representation_score(model, test_tokens, rv, best_hook)

token_strs = [tokenizer.decode([int(t)]) for t in test_tokens]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(token_strs)), scores['scores'], color='steelblue')
ax.set_xticks(range(len(token_strs)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_ylabel('Concept Score')
ax.set_title('Per-Token Concept Presence')
plt.tight_layout()
plt.show()

## 4. Control Vector Intervention

Steer model behavior by adding reading vectors during inference.

In [ ]:
test_tokens = model.to_tokens("The capital of the country is")
vecs = {best_hook: rv}

result = repe.control_vector_intervention(model, test_tokens, vecs, coefficient=2.0)

print("Top tokens most affected by steering:")
for tok_idx, change in result['top_changed_tokens'][:10]:
    token_str = tokenizer.decode([tok_idx])
    print(f"  '{token_str}': logit change = {change:+.4f}")

## 5. Concept Suppression Curve

How does the metric change as we sweep the control vector coefficient?

In [ ]:
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

tokens = model.to_tokens("The Eiffel Tower is located in")
result = repe.concept_suppression_curve(
    model, tokens, vecs, metric,
    coefficients=[-5, -3, -1, 0, 1, 3, 5]
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(result['coefficients'], result['metrics'], 'bo-')
ax.axhline(result['baseline_metric'], color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Control Vector Coefficient')
ax.set_ylabel('Paris Logit')
ax.set_title(f'Concept Suppression Curve (sensitivity: {result["sensitivity"]:.4f})')
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `extract_reading_vectors()` | Contrastive PCA to find concept direction |
| `reading_vector_scan()` | Layer-by-layer concept encoding profile |
| `control_vector_intervention()` | Steer model with reading vectors |
| `representation_score()` | Per-token concept presence scoring |
| `concept_suppression_curve()` | Metric response to steering coefficient |